# Базовый дискретно-событийный эксперимент SIR

Этот сценарий строит дискретно-событийную модель распространения инфекции SIR,
в которой каждый индивид представлен отдельным процессом ConcurrentSim. Прогон
сохраняет временной ряд численности `S`, `I`, `R`, сводные метрики, ансамбль
независимых стохастических реализаций и детерминированную систему ОДУ для
сравнения, а также формирует графики.

In [1]:
using DrWatson
@quickactivate "project"

using CSV
using CairoMakie

include(srcdir("SIRModels.jl"))
using .SIRModels

mkpath(datadir())
mkpath(plotsdir())

"/home/lilidji/labs/lab08/project/plots"

## Параметры модели

Начальное состояние популяции `u0 = [S0, I0, R0]` и параметры `p = [β, c, γ]`:
вероятность передачи при контакте, частота контактов и интенсивность
выздоровления. Базовое репродуктивное число `R0 = β·c/γ = 2.0`.

In [2]:
u0 = [990, 10, 0]
p = [0.05, 10.0, 0.25]
tmax = 40.0
seed = 1234

1234

## Базовый прогон

Один прогон дискретно-событийной модели и его сводные характеристики.

In [3]:
des = run_sir_des(u0, p, tmax; seed = seed)
summary = sir_summary(des, u0, p)

Row,beta,c,gamma,N,r0,peak_I,peak_time,final_R,final_fraction,analytic_final_fraction,events
,Float64,Float64,Float64,Int64,Float64,Int64,Float64,Int64,Float64,Float64,Int64
1,0.05,10.0,0.25,1000,2.0,152,18.4475,759,0.759,0.796812,1519


## Детерминированное сравнение

Та же система в виде ОДУ, проинтегрированная методом Рунге–Кутты 4-го порядка.

In [4]:
ode = run_sir_ode(u0, p, tmax; dt = 0.05)

Row,t,S,I,R
,Float64,Float64,Float64,Float64
1,0.0,990.0,10.0,0.0
2,0.05,989.751,10.1232,0.125769
3,0.1,989.499,10.2479,0.253087
4,0.15,989.244,10.374,0.381972
5,0.2,988.986,10.5017,0.512444
6,0.25,988.725,10.6308,0.64452
7,0.3,988.46,10.7615,0.77822
8,0.35,988.193,10.8936,0.913563
9,0.4,987.922,11.0274,1.05057


## Ансамбль стохастических реализаций

Набор независимых прогонов с разными зёрнами позволяет увидеть разброс
траекторий и оценить вероятность раннего затухания эпидемии.

In [5]:
trajectories, ensemble = run_sir_ensemble(u0, p, tmax, 24; seed = 1000)

(DataFrames.DataFrame[1566×4 DataFrame
  Row │ t          S      I      R     
      │ Float64    Int64  Int64  Int64 
──────┼────────────────────────────────
    1 │  0.0         990     10      0
    2 │  0.182131    989     11      0
    3 │  0.183757    989     10      1
    4 │  0.266518    989      9      2
    5 │  0.473027    988     10      2
    6 │  0.617593    987     11      2
    7 │  0.923556    987     10      3
    8 │  1.03525     986     11      3
  ⋮   │     ⋮        ⋮      ⋮      ⋮
 1560 │ 38.3546      208     15    777
 1561 │ 38.4839      207     16    777
 1562 │ 38.6158      206     17    777
 1563 │ 38.8473      206     16    778
 1564 │ 38.8597      206     15    779
 1565 │ 39.2316      206     14    780
 1566 │ 39.9219      206     13    781
                      1551 rows omitted, 1624×4 DataFrame
  Row │ t            S      I      R     
      │ Float64      Int64  Int64  Int64 
──────┼──────────────────────────────────
    1 │  0.0           990     10  

## Сохранение результатов

In [6]:
CSV.write(datadir("sir_trajectory.csv"), des)
CSV.write(datadir("sir_ode.csv"), ode)
CSV.write(datadir("sir_summary.csv"), summary)
CSV.write(datadir("sir_ensemble.csv"), ensemble)

"/home/lilidji/labs/lab08/project/data/sir_ensemble.csv"

## Графики

In [7]:
save(plotsdir("sir_trajectory.png"), plot_sir_trajectory(des))
save(plotsdir("sir_des_vs_ode.png"), plot_sir_des_vs_ode(des, ode))
save(plotsdir("sir_ensemble.png"), plot_sir_ensemble(trajectories, ode))
save(plotsdir("sir_phase.png"), plot_sir_phase(des))

## Оценка производительности

Замер времени одного прогона для популяций разного размера показывает, как
растёт стоимость симуляции с числом агентов.

In [8]:
for n in (1000, 2000)
    local_u0 = [n - 10, 10, 0]
    elapsed = @elapsed run_sir_des(local_u0, p, tmax; seed = seed)
    println("sir_run for N=$(n): $(round(elapsed, digits = 3)) s")
end

println("SIR baseline experiment completed")
println(summary)

sir_run for N=1000: 0.379 s
sir_run for N=2000: 1.215 s
SIR baseline experiment completed
1×11 DataFrame
 Row │ beta     c        gamma    N      r0       peak_I  peak_time  final_R  final_fraction  analytic_final_fraction  events
     │ Float64  Float64  Float64  Int64  Float64  Int64   Float64    Int64    Float64         Float64                  Int64
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │    0.05     10.0     0.25   1000      2.0     152    18.4475      759           0.759                 0.796812    1519
